# S0 — 01.05: Gradient Descent: Taking Steps Down a Loss Landscape

**Module:** 01 — ML Fundamentals  
**Prerequisites:** 01.01–01.04  
**Difficulty:** introductory  
**Estimated time:** 45–60 minutes

> **Learner question:** How can a parameter update use the slope of a loss to move toward a better value?

This is a deterministic, synthetic, dimensionless demonstration. It uses one scalar parameter and exactly three updates. It does not claim convergence, performance, or generalisation.

## S1 — Fixed model and data

There is no intercept. The prediction rule is:

$$\hat y_i(w)=w x_i$$

We use `x = [1.0, 2.0]`, `y = [2.0, 4.0]`, start at `w_0 = 0.0`, and use learning rate `eta = 0.1`.

In [ ]:
import numpy as np

RTOL = 0.0
ATOL = 1e-12
x = np.array([1.0, 2.0], dtype=np.float64)
y = np.array([2.0, 4.0], dtype=np.float64)
w = np.float64(0.0)
eta = np.float64(0.1)
x, y, w, eta

## S2 — Intuition: the loss landscape

The gradient is the local slope of the objective. A negative gradient means the loss decreases locally when `w` increases, so subtracting the gradient moves rightward. The learning rate scales the move; it is not the loss or the gradient.

## S3 — Objective and residuals

Use residual = observed − predicted:

$$r_i(w)=y_i-wx_i$$

$$J(w)=\frac12\sum_i (y_i-wx_i)^2$$

Because this fixed example has two observations, `1/2` is also its ordinary mean factor. At `w=0`, residuals are `[2.0, 4.0]`, so `J(0) = (2² + 4²)/2 = 10.0`.

In [ ]:
def predict(x, w):
    return w * x


def objective(x, y, w):
    residuals = y - predict(x, w)
    return np.float64(0.5 * np.sum(residuals**2, dtype=np.float64))


residuals = y - predict(x, w)
print("predictions:", predict(x, w))
print("residuals:", residuals)
print("J(0):", objective(x, y, w))

## S4 — Derivative and update rule

The chain rule gives:

$$\frac{dJ}{dw}=\frac12\sum_i 2(y_i-wx_i)(-x_i)=-\sum_i x_i(y_i-wx_i)$$

The `2/2` is essential. For these data, the gradient is `5w − 10`. Gradient descent uses:

$$w_{\mathrm{new}}=w-\eta\frac{dJ}{dw}$$

At `w=0`, the gradient is `-10.0`, the update amount is `+1.0`, and the next parameter is `w_1=1.0`.

In [ ]:
def gradient(x, y, w):
    residuals = y - w * x
    return np.float64(-np.sum(x * residuals, dtype=np.float64))


def update(w, gradient_value, eta):
    return np.float64(w - eta * gradient_value)


slope = gradient(x, y, w)
next_w = update(w, slope, eta)
print("gradient:", slope)
print("update amount:", next_w - w)
print("next parameter:", next_w)

## S5–S6 — Hand update and deterministic NumPy implementation

The loss is a scalar value, the gradient is a signed slope, and `-eta * gradient` is the parameter change. The loop below performs exactly three updates.

In [ ]:
rows = []
w = np.float64(0.0)
for step in range(3):
    loss = objective(x, y, w)
    slope = gradient(x, y, w)
    next_w = update(w, slope, eta)
    rows.append((step, w, loss, slope, next_w))
    w = next_w

print("step | current w | J(w) | gradient | next w")
for row in rows:
    print(f"{row[0]}    | {row[1]:9.2f} | {row[2]:4.3f} | {row[3]:8.1f} | {row[4]:6.2f}")

The frozen path is:

| step | current `w` | `J(w)` | gradient | next `w` |
|---:|---:|---:|---:|---:|
| 0 | `0.0` | `10.0` | `-10.0` | `1.0` |
| 1 | `1.0` | `2.5` | `-5.0` | `1.5` |
| 2 | `1.5` | `0.625` | `-2.5` | `1.75` |

After the third update, `w_3 = 1.75` and `J(w_3) = 0.15625`.

## S7 — Verification and exact minimizer

For this fixed objective:

$$\frac{dJ}{dw}=5w-10=0\quad\Rightarrow\quad w^*=2.0$$

The finite path approaches the exact minimizer but does not reach it in three updates. The following checks use zero relative tolerance and absolute tolerance `1e-12`.

In [ ]:
expected_w = np.array([0.0, 1.0, 1.5, 1.75], dtype=np.float64)
observed_w = np.array([0.0, rows[0][4], rows[1][4], rows[2][4]], dtype=np.float64)
expected_losses = np.array([10.0, 2.5, 0.625, 0.15625], dtype=np.float64)
observed_losses = np.array([objective(x, y, value) for value in expected_w], dtype=np.float64)
np.testing.assert_allclose(observed_w, expected_w, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(observed_losses, expected_losses, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(5.0 * 2.0 - 10.0, 0.0, rtol=RTOL, atol=ATOL)
print("verified: finite endpoint w_3=1.75, J(w_3)=0.15625, exact w*=2.0")

## S8 — Static visual

![Gradient descent visual](assets/gradient_descent_visual.png)

**Caption:** A one-dimensional synthetic loss curve and update-flow table show three deterministic gradient-descent steps from `w=0.0` to `w=1.75` with `eta=0.1`; the exact minimizer `w*=2.0` is marked separately.

**Description:** The left panel shows the objective `J(w)` against dimensionless `w`. A vermillion solid line with circular markers and arrowheads traces `0.0 -> 1.0 -> 1.5 -> 1.75`; a purple dashed line separately marks the exact minimizer `w*=2.0`. The right panel lists each current parameter, objective, gradient, and next parameter, followed by the update rule. The footer discloses the synthetic, dimensionless data and that three updates do not establish a convergence guarantee.

## S9 — Failure modes and bounded learning-rate discussion

- A wrong sign, `w + eta * gradient`, moves uphill here.
- A wrong derivative scale would give `-20.0` at `w=0`, contradicting the stated half-sum objective.
- A much larger learning rate can overshoot or oscillate; this lesson does not give a universal safe range.
- Three decreasing losses are evidence about this finite calculation, not a convergence proof or a generalisation guarantee.

### S10 — Exercises

1. At `w=1.0`, calculate the residuals, loss, gradient, and next parameter.
2. Explain why the update from `w=0.0` moves to the right.
3. Identify the loss, gradient, update amount, and next parameter in step 0.
4. Solve `dJ/dw=0` and compare the exact minimizer with `w_3`.
5. Explain why three fixed updates cannot guarantee behavior on other data.

## S11 — References and continuity

- Goodfellow, Bengio, and Courville, *Deep Learning*, §4.3.
- Deisenroth, Faisal, and Ong, *Mathematics for Machine Learning*, Chapter 9.
- Bishop, *Pattern Recognition and Machine Learning*, §5.1.
- NumPy 1.26 documentation for array arithmetic and testing.
- Matplotlib documentation for static plotting and annotations.

This lesson builds on prediction and loss vocabulary from 01.01–01.04. It is a scalar optimization demonstration, not a train/validation/test procedure or an introduction to other optimizers.

## S12 — Takeaways and boundary check

Gradient descent subtracts a learning-rate-scaled gradient:

$$w_{new}=w-\eta\frac{dJ}{dw}$$

For this fixed synthetic example, the path is `0.0 -> 1.0 -> 1.5 -> 1.75`, and the exact minimizer is `2.0`. The lesson stops at three deterministic updates. It does not teach stochastic gradient descent, minibatches, momentum, Adam, automatic differentiation, regularization, hyperparameter search, neural networks, convergence proofs, deployment, performance claims, or generalisation guarantees.